In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/adithyadasaadhiii@gmail.com/consolidated_pipeline/01_setup/utilities

In [0]:
print(silver_schema,bronze_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sports-bar-raw/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.
        format('csv')
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp",F.current_timestamp())
        .select("*","_metadata.file_name","_metadata.file_size")
)
display(df)

In [0]:
df.write.mode("overwrite").option("changeDataFeed", "true").saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

##Silver Processing##

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")
df_duplicates.show()

In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])

In [0]:
df_silver.groupBy("customer_id").count().filter("count > 1").show()

In [0]:
display(
    df_silver.filter(
        F.col("customer_name") != F.trim(F.col("customer_name"))
        )
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)
df_silver.select("customer_name").show()

In [0]:
df_silver.select(
    F.length("customer_name").alias("len"),
    F.concat(F.lit("["), F.col("customer_name"), F.lit("]")).alias("value")
).show(truncate=False)

In [0]:
display(
    df_silver.filter(
        F.col("customer_name") == F.trim(F.col("customer_name"))
        )
)


In [0]:
df_silver.select("city").distinct().show()

In [0]:
#typos -> correct name

city_mapping = {
    "Bengalore" : "Bengaluru",
    "Bengaluruu" : "Bengaluru",

    "Hyderabadd" :  "Hyderabad",
    "Hyderbad" : "Hyderabad",

    "NewDelhee" : "New Delhi",
    "NewDelhi" : "New Delhi",
    "NewDheli" : "New Delhi"
}

allowed = ["Bengaluru","Hyderabad","New Delhi"]

df_silver = (
    df_silver.replace(city_mapping,subset="city")
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
        
    )
)
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter("City is null").show()

In [0]:
df_silver.select("customer_name").distinct().orderBy("customer_name").show()

In [0]:
city_null_customer_name = ["Sprintx Nutrition","Zenathlete Foods","Primefuel Nutrition","Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(city_null_customer_name)).show()


In [0]:
customer_city_fix = {
    789403 : "New Delhi",
    789420 : "Bengaluru",
    789521 : "Hyderabad",
    789603 : "Hyderabad",
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
        .join(df_fix,on="customer_id",how="left")
        .withColumn(
            "city",
            F.coalesce(F.col("city"),F.col("fixed_city"))
        )
        .drop("fixed_city")
)
display(df_silver)

In [0]:
df_silver = (
    df_silver
        .withColumn(
            "customer",
            F.concat_ws( "-",F.col("customer_name"),F.coalesce(F.col("city"),F.lit("Unknown")))
        )
        .withColumn("market",F.lit("India"))
        .withColumn("platform",F.lit("Sports Bar"))
        .withColumn("channel",F.lit("Acquisition"))
)
display(df_silver)

In [0]:
df_silver.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true")\
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##Gold Processing##

In [0]:
df_silver = spark.read.table(f"{catalog}.{silver_schema}.{data_source}")

df_gold = (
    df_silver.selectExpr("cast(customer_id as string) as customer_code","customer_name","city","customer","market","platform","channel")   
)

display(df_gold)

In [0]:
df_gold.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true")\
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

##Merging with the parent table##

In [0]:
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
  "customer_code",
  "customer",
  "market",
  "platform",
  "channel"
)

In [0]:
delta_table.alias("target").merge(
    source = df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()